<a href="https://colab.research.google.com/github/Watchman77/AR_Pure_SolarFlares_72h_Dataset/blob/main/XFlareXAI_01_Build_48h_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# X-FlareXAI 01: Build 48-Hour SHARP + GOES Dataset

**Purpose:** Generate a leakage-safe 48-hour M/X-class solar flare label (`label_MX_48h`) from SHARP active-region snapshots and GOES X-ray flare event catalogues.

**Output:** `HMI_AR_2010_2025_ML_READY_48H.csv`

This notebook prepares the foundation dataset for the X-FlareXAI paper:

> **X-FlareXAI: Comparative Explainable Artificial Intelligence for 48-Hour Solar Flare Forecasting Using SHARP Magnetic and GOES Flare-History Features**


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# X-FlareXAI 01
# Build 48-Hour SHARP + GOES Solar Flare Forecasting Dataset
# ============================================================

import os
import re
import glob
from datetime import timedelta

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Libraries loaded successfully.")


Libraries loaded successfully.


In [3]:
# ============================================================
# 1. Configure paths
# ============================================================

# Change this if your project folder is different.
BASE_PATH = "/content/drive/MyDrive/AR_Stratified/CICCE_BRADFORD_PROJECT"

# Input SHARP CSV.
SHARP_PATH = os.path.join(BASE_PATH, "HMI_AR_2010_2025_CLEAN_FRESH.csv")

# Folder containing GOES text files:
# GOES XRAY events for 2010.txt, ..., GOES XRAY events for 2025.txt
GOES_FOLDER = os.path.join(BASE_PATH, "GOES_XRAY_EVENTS")

# Output dataset.
OUTPUT_PATH = os.path.join(BASE_PATH, "HMI_AR_2010_2025_ML_READY_48H.csv")

print("BASE_PATH:", BASE_PATH)
print("SHARP_PATH:", SHARP_PATH)
print("GOES_FOLDER:", GOES_FOLDER)
print("OUTPUT_PATH:", OUTPUT_PATH)


BASE_PATH: /content/drive/MyDrive/AR_Stratified/CICCE_BRADFORD_PROJECT
SHARP_PATH: /content/drive/MyDrive/AR_Stratified/CICCE_BRADFORD_PROJECT/HMI_AR_2010_2025_CLEAN_FRESH.csv
GOES_FOLDER: /content/drive/MyDrive/AR_Stratified/CICCE_BRADFORD_PROJECT/GOES_XRAY_EVENTS
OUTPUT_PATH: /content/drive/MyDrive/AR_Stratified/CICCE_BRADFORD_PROJECT/HMI_AR_2010_2025_ML_READY_48H.csv


In [4]:
# ============================================================
# 2. Load and clean SHARP active-region data
# ============================================================

sharp_df = pd.read_csv("/content/drive/MyDrive/AR_Stratified/HMI_SHARP/HMI_AR_2010_2025_CLEAN_FRESH.csv")

print("Original SHARP shape:", sharp_df.shape)
display(sharp_df.head())

# Ensure timestamp exists.
if "T_REC_dt" not in sharp_df.columns:
    raise ValueError("Expected column 'T_REC_dt' not found in SHARP file.")

# Ensure NOAA_AR exists.
if "NOAA_AR" not in sharp_df.columns:
    raise ValueError("Expected column 'NOAA_AR' not found in SHARP file.")

sharp_df["T_REC_dt"] = pd.to_datetime(sharp_df["T_REC_dt"], errors="coerce")

# Clean NOAA_AR.
sharp_df["NOAA_AR"] = pd.to_numeric(sharp_df["NOAA_AR"], errors="coerce")

sharp_df = sharp_df.dropna(subset=["T_REC_dt", "NOAA_AR"]).copy()
sharp_df["NOAA_AR"] = sharp_df["NOAA_AR"].astype(int)

# Sort for safe temporal processing.
sharp_df = sharp_df.sort_values(["NOAA_AR", "T_REC_dt"]).reset_index(drop=True)

print("Clean SHARP shape:", sharp_df.shape)
print("Date range:", sharp_df["T_REC_dt"].min(), "to", sharp_df["T_REC_dt"].max())
print("Unique NOAA ARs:", sharp_df["NOAA_AR"].nunique())

display(sharp_df[["NOAA_AR", "T_REC_dt"]].head())


Original SHARP shape: (26604, 31)


,T_REC,MEANGBZ,HARPNUM,R_VALUE,TOTUSJH,USFLUX,TOTPOT,MEANPOT,AREA_ACR,NOAA_ARS,LON_MIN,LON_MAX,LAT_MIN,LAT_MAX,QUALITY,USFLUX.1,MEANGAM,MEANGBT,MEANGBH,MEANJZD,TOTUSJZ,MEANALP,MEANJZH,ABSNJZH,SAVNCPP,MEANSHR,SHRGT45,T_REC_dt,year,NOAA_AR,AREA_ACR.1
0,2010.05.01_12:00:00_TAI,82.596,1,2.960,263.431,5.785536e+21,1.720008e+22,1496.756,66.219543,11067,-75.864128,-67.234299,21.990669,25.595324,0,5.785536e+21,23.760,75.801,32.039,-0.202332,5.041179e+12,-0.003394,-0.001074,9.291,9.349135e+11,18.650,3.583,2010-05-01 12:00:00,2010,11067.0,NaN
1,2010.05.01_12:00:00_TAI,135.426,2,0.000,108.818,1.230051e+21,7.219153e+21,2577.490,102.988136,11064,-25.696795,-17.056698,10.415223,18.482346,0,1.230051e+21,34.823,136.321,66.673,0.757142,2.411247e+12,-0.011185,-0.003109,6.553,8.526952e+11,26.338,8.725,2010-05-01 12:00:00,2010,11064.0,NaN
2,2010.05.01_12:00:00_TAI,132.409,5,0.000,26.725,4.299636e+20,1.324046e+21,1388.564,45.107292,MISSING,9.501516,18.729277,-29.886215,-27.243484,0,4.299636e+20,27.766,133.080,52.031,0.115033,6.365679e+11,0.001626,0.000374,0.268,4.410475e+10,20.070,0.696,2010-05-01 12:00:00,2010,NaN,NaN
3,2010.05.01_12:00:00_TAI,128.109,6,0.000,41.505,6.926417e+20,2.407468e+21,1488.334,67.645287,11065,1.309394,8.360654,-34.004143,-29.698412,0,6.926417e+20,29.002,129.000,49.164,0.375635,1.021202e+12,-0.007351,-0.001561,1.901,2.443167e+11,22.353,3.612,2010-05-01 12:00:00,2010,11065.0,NaN
4,2010.05.02_12:00:00_TAI,82.971,1,3.173,486.936,1.024649e+22,3.510334e+22,1719.177,203.945465,11067,-64.320930,-52.247795,20.969666,27.029732,0,1.024649e+22,26.171,81.016,34.770,-0.223050,9.660283e+12,0.002879,0.000917,14.103,1.832217e+12,20.957,3.668,2010-05-02 12:00:00,2010,11067.0,NaN


Clean SHARP shape: (22355, 31)
Date range: 2010-05-01 12:00:00 to 2025-07-16 12:00:00
Unique NOAA ARs: 2268


,NOAA_AR,T_REC_dt
0,11063,2010-05-03 12:00:00
1,11063,2010-05-04 12:00:00
2,11063,2010-05-05 12:00:00
3,11064,2010-05-01 12:00:00
4,11064,2010-05-02 12:00:00


In [ ]:
# ============================================================
# 3. Parse GOES X-ray event text files
# ============================================================

goes_files = sorted(glob.glob(os.path.join(GOES_FOLDER, "*.txt")))
print("GOES files found:", len(goes_files))

for f in goes_files[:5]:
    print(" -", os.path.basename(f))

if len(goes_files) == 0:
    raise FileNotFoundError(
        f"No GOES .txt files found in {GOES_FOLDER}. "
        "Please check the folder path and file names."
    )


def flare_class_to_flux(class_str):
    '''
    Convert GOES flare class to approximate peak flux scale.

    A1.0 = 1e-8
    B1.0 = 1e-7
    C1.0 = 1e-6
    M1.0 = 1e-5
    X1.0 = 1e-4
    '''
    if not isinstance(class_str, str) or len(class_str) < 2:
        return np.nan

    class_str = class_str.strip().upper()
    letter = class_str[0]

    try:
        value = float(class_str[1:])
    except Exception:
        return np.nan

    scale = {
        "A": 1e-8,
        "B": 1e-7,
        "C": 1e-6,
        "M": 1e-5,
        "X": 1e-4,
    }

    return value * scale.get(letter, np.nan)


def parse_goes_file(path):
    '''
    Parse one GOES yearly event list.

    Expected row format:
    Date Start Peak End Class Position ActiveRegion

    Example:
    1-Jan-2014 18:40 18:52 19:03 M9.9 S16W45 11936
    '''
    records = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    for raw in lines:
        line = raw.strip()

        # Skip headers and short/empty lines.
        if (
            not line
            or "Written" in line
            or "events were retrieved" in line
            or "Columns:" in line
            or len(line) < 20
        ):
            continue

        parts = line.split()

        # Minimum required: date, start, peak, end, class.
        if len(parts) < 5:
            continue

        date_str = parts[0]
        start_str = parts[1]
        peak_str = parts[2]
        end_str = parts[3]
        flare_class = parts[4].upper().strip()

        # Last token is active region when available; otherwise it may be position or missing.
        ar_token = parts[-1].strip()
        if not ar_token.isdigit():
            continue

        try:
            noaa_ar = int(ar_token)
        except Exception:
            continue

        # Parse start time.
        try:
            start_time = pd.to_datetime(
                f"{date_str} {start_str}",
                format="%d-%b-%Y %H:%M",
                errors="raise"
            )
        except Exception:
            continue

        # Peak and end may cross midnight; parse approximately using date_str.
        peak_time = pd.to_datetime(f"{date_str} {peak_str}", format="%d-%b-%Y %H:%M", errors="coerce")
        end_time = pd.to_datetime(f"{date_str} {end_str}", format="%d-%b-%Y %H:%M", errors="coerce")

        # If peak/end appears earlier than start, assume next day.
        if pd.notna(peak_time) and peak_time < start_time:
            peak_time += pd.Timedelta(days=1)

        if pd.notna(end_time) and end_time < start_time:
            end_time += pd.Timedelta(days=1)

        records.append({
            "flare_start_time": start_time,
            "flare_peak_time": peak_time,
            "flare_end_time": end_time,
            "flare_class": flare_class,
            "flare_flux": flare_class_to_flux(flare_class),
            "NOAA_AR": noaa_ar,
            "source_file": os.path.basename(path)
        })

    return records


all_records = []
for path in tqdm(goes_files, desc="Parsing GOES files"):
    all_records.extend(parse_goes_file(path))

goes_df = pd.DataFrame(all_records)

if goes_df.empty:
    raise ValueError("GOES parsing produced an empty dataframe. Please inspect the input text files.")

goes_df = goes_df.dropna(subset=["flare_start_time", "flare_class", "NOAA_AR"]).copy()
goes_df["NOAA_AR"] = goes_df["NOAA_AR"].astype(int)
goes_df = goes_df.sort_values(["NOAA_AR", "flare_start_time"]).reset_index(drop=True)

print("Parsed GOES shape:", goes_df.shape)
print("GOES date range:", goes_df["flare_start_time"].min(), "to", goes_df["flare_start_time"].max())
print("Unique GOES NOAA ARs:", goes_df["NOAA_AR"].nunique())
print(goes_df["flare_class"].str[0].value_counts().sort_index())

display(goes_df.head())


GOES files found: 16
 - GOES XRAY events for 2010.txt
 - GOES XRAY events for 2011.txt
 - GOES XRAY events for 2012.txt
 - GOES XRAY events for 2013.txt
 - GOES XRAY events for 2014.txt


Parsing GOES files:   0%|          | 0/16 [00:00<?, ?it/s]

Parsed GOES shape: (27667, 7)
GOES date range: 2010-01-01 12:02:00 to 2025-04-18 12:03:00
Unique GOES NOAA ARs: 2047
flare_class
A      585
B     7463
C    17379
M     2128
X      112
Name: count, dtype: int64


,flare_start_time,flare_peak_time,flare_end_time,flare_class,flare_flux,NOAA_AR,source_file
0,2010-01-01 12:02:00,2010-01-01 12:09:00,2010-01-01 12:18:00,B1.9,1.900000e-07,11039,GOES XRAY events for 2010.txt
1,2010-01-01 12:33:00,2010-01-01 12:43:00,2010-01-01 13:00:00,B2.3,2.300000e-07,11039,GOES XRAY events for 2010.txt
2,2010-01-01 23:29:00,2010-01-01 23:33:00,2010-01-01 23:42:00,B1.1,1.100000e-07,11039,GOES XRAY events for 2010.txt
3,2010-01-02 03:10:00,2010-01-02 03:13:00,2010-01-02 03:19:00,B1.1,1.100000e-07,11039,GOES XRAY events for 2010.txt
4,2010-01-02 07:58:00,2010-01-02 08:05:00,2010-01-02 08:13:00,B6.4,6.400000e-07,11039,GOES XRAY events for 2010.txt


In [ ]:
# ============================================================
# 4. Generate AR-specific 48-hour M/X labels
# ============================================================

# Keep M/X events for future label construction.
mx_df = goes_df[goes_df["flare_class"].str.startswith(("M", "X"), na=False)].copy()

print("Total M/X GOES events with NOAA AR:", len(mx_df))
display(mx_df.head())

# Create fast lookup dictionary by NOAA_AR.
mx_by_ar = {
    ar: group["flare_start_time"].sort_values().to_numpy()
    for ar, group in mx_df.groupby("NOAA_AR")
}

def has_future_mx_flare(ar, obs_time, horizon_hours=48):
    '''
    Return 1 if same NOAA AR has M/X flare in (obs_time, obs_time + horizon].
    '''
    times = mx_by_ar.get(ar)

    if times is None or len(times) == 0:
        return 0

    start = np.datetime64(obs_time)
    end = np.datetime64(obs_time + pd.Timedelta(hours=horizon_hours))

    # Find first flare strictly after obs_time.
    left = np.searchsorted(times, start, side="right")
    right = np.searchsorted(times, end, side="right")

    return int(right > left)

tqdm.pandas(desc="Creating label_MX_48h")

sharp_df["label_MX_48h"] = sharp_df.progress_apply(
    lambda row: has_future_mx_flare(row["NOAA_AR"], row["T_REC_dt"], horizon_hours=48),
    axis=1
)

print("48h label distribution:")
display(sharp_df["label_MX_48h"].value_counts().rename("count").to_frame())

print("48h label distribution (%):")
display((sharp_df["label_MX_48h"].value_counts(normalize=True) * 100).rename("percent").to_frame())


Total M/X GOES events with NOAA AR: 2240


,flare_start_time,flare_peak_time,flare_end_time,flare_class,flare_flux,NOAA_AR,source_file
27,2010-01-20 10:46:00,2010-01-20 10:59:00,2010-01-20 11:10:00,M1.8,0.000018,11041,GOES XRAY events for 2010.txt
58,2010-02-07 02:20:00,2010-02-07 02:34:00,2010-02-07 02:39:00,M6.4,0.000064,11045,GOES XRAY events for 2010.txt
72,2010-02-08 11:57:00,2010-02-08 12:03:00,2010-02-08 12:06:00,M1.1,0.000011,11045,GOES XRAY events for 2010.txt
104,2010-02-12 11:19:00,2010-02-12 11:26:00,2010-02-12 11:28:00,M8.3,0.000083,11046,GOES XRAY events for 2010.txt
106,2010-02-12 17:52:00,2010-02-12 18:08:00,2010-02-12 18:08:00,M1.1,0.000011,11046,GOES XRAY events for 2010.txt


Creating label_MX_48h:   0%|          | 0/22355 [00:00<?, ?it/s]

48h label distribution:


,count
label_MX_48h,
0,21187
1,1168


48h label distribution (%):


,percent
label_MX_48h,
0,94.775218
1,5.224782


In [ ]:
# ============================================================
# 5. Generate GOES flare-history features
# ============================================================

# Create lookup dictionaries for all GOES events by NOAA_AR.
goes_by_ar = {
    ar: group.sort_values("flare_start_time").reset_index(drop=True)
    for ar, group in goes_df.groupby("NOAA_AR")
}

def flare_history_features(ar, obs_time):
    '''
    Compute causal flare-history features using only events before obs_time.
    '''
    group = goes_by_ar.get(ar)

    if group is None or group.empty:
        return pd.Series({
            "MX_count_last_24h": 0,
            "MX_count_last_48h": 0,
            "C_count_last_24h": 0,
            "C_count_last_48h": 0,
            "max_flare_flux_last_48h": 0.0,
            "mean_flare_flux_last_48h": 0.0,
            "time_since_last_MX_hours": 9999.0,
            "flare_activity_index_48h": 0.0
        })

    t = pd.Timestamp(obs_time)
    past_24_start = t - pd.Timedelta(hours=24)
    past_48_start = t - pd.Timedelta(hours=48)

    past_24 = group[
        (group["flare_start_time"] < t) &
        (group["flare_start_time"] >= past_24_start)
    ]

    past_48 = group[
        (group["flare_start_time"] < t) &
        (group["flare_start_time"] >= past_48_start)
    ]

    mx_24 = past_24[past_24["flare_class"].str.startswith(("M", "X"), na=False)]
    mx_48 = past_48[past_48["flare_class"].str.startswith(("M", "X"), na=False)]
    c_24 = past_24[past_24["flare_class"].str.startswith("C", na=False)]
    c_48 = past_48[past_48["flare_class"].str.startswith("C", na=False)]

    if len(mx_48) > 0:
        last_mx_time = mx_48["flare_start_time"].max()
        time_since_last_mx = (t - last_mx_time).total_seconds() / 3600
    else:
        # Use large sentinel meaning no recent M/X flare.
        time_since_last_mx = 9999.0

    flux_48 = past_48["flare_flux"].dropna()

    # Weighted activity index using approximate GOES flux.
    flare_activity_index = flux_48.sum() if len(flux_48) > 0 else 0.0

    return pd.Series({
        "MX_count_last_24h": int(len(mx_24)),
        "MX_count_last_48h": int(len(mx_48)),
        "C_count_last_24h": int(len(c_24)),
        "C_count_last_48h": int(len(c_48)),
        "max_flare_flux_last_48h": float(flux_48.max()) if len(flux_48) > 0 else 0.0,
        "mean_flare_flux_last_48h": float(flux_48.mean()) if len(flux_48) > 0 else 0.0,
        "time_since_last_MX_hours": float(time_since_last_mx),
        "flare_activity_index_48h": float(flare_activity_index)
    })


tqdm.pandas(desc="Creating GOES history features")

history_df = sharp_df.progress_apply(
    lambda row: flare_history_features(row["NOAA_AR"], row["T_REC_dt"]),
    axis=1
)

sharp_48h_df = pd.concat([sharp_df.reset_index(drop=True), history_df.reset_index(drop=True)], axis=1)

print("Dataset with history features:", sharp_48h_df.shape)
display(sharp_48h_df.head())


Creating GOES history features:   0%|          | 0/22355 [00:00<?, ?it/s]

Dataset with history features: (22355, 40)


,T_REC,MEANGBZ,HARPNUM,R_VALUE,TOTUSJH,USFLUX,TOTPOT,MEANPOT,AREA_ACR,NOAA_ARS,LON_MIN,LON_MAX,LAT_MIN,LAT_MAX,QUALITY,USFLUX.1,MEANGAM,MEANGBT,MEANGBH,MEANJZD,TOTUSJZ,MEANALP,MEANJZH,ABSNJZH,SAVNCPP,MEANSHR,SHRGT45,T_REC_dt,year,NOAA_AR,AREA_ACR.1,label_MX_48h,MX_count_last_24h,MX_count_last_48h,C_count_last_24h,C_count_last_48h,max_flare_flux_last_48h,mean_flare_flux_last_48h,time_since_last_MX_hours,flare_activity_index_48h
0,2010.05.03_12:00:00_TAI,163.413,11,2.856,78.875,1.147928e+21,8.805230e+21,6082.771,37.772018,11063,44.184227,49.759857,16.405930,18.883835,65536,1.147928e+21,33.823,157.335,82.652,0.853483,1.032882e+12,0.022914,0.018148,19.782,6.899534e+11,24.717,4.954,2010-05-03 12:00:00,2010,11063,NaN,0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,9999.0,0.000000e+00
1,2010.05.04_12:00:00_TAI,114.931,11,0.000,85.789,1.960732e+21,6.842635e+21,2586.554,45.436054,11063,56.653294,63.360962,16.160620,19.021460,65536,1.960732e+21,23.572,111.969,48.270,-0.446712,1.144288e+12,-0.006346,-0.003990,7.947,4.758987e+11,17.322,0.000,2010-05-04 12:00:00,2010,11063,NaN,0,0.0,0.0,0.0,0.0,1.500000e-07,1.500000e-07,9999.0,1.500000e-07
2,2010.05.05_12:00:00_TAI,87.534,11,2.192,250.729,6.231462e+21,3.177784e+22,2972.089,108.927094,11063,68.908920,78.151360,15.653858,19.510757,0,6.231462e+21,28.006,79.667,40.057,-0.065026,4.168505e+12,-0.004862,-0.002102,16.925,2.801237e+11,22.145,3.130,2010-05-05 12:00:00,2010,11063,NaN,0,0.0,0.0,0.0,0.0,1.500000e-07,1.500000e-07,9999.0,1.500000e-07
3,2010.05.01_12:00:00_TAI,135.426,2,0.000,108.818,1.230051e+21,7.219153e+21,2577.490,102.988136,11064,-25.696795,-17.056698,10.415223,18.482346,0,1.230051e+21,34.823,136.321,66.673,0.757142,2.411247e+12,-0.011185,-0.003109,6.553,8.526952e+11,26.338,8.725,2010-05-01 12:00:00,2010,11064,NaN,0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,9999.0,0.000000e+00
4,2010.05.02_12:00:00_TAI,129.025,2,0.000,42.533,5.602466e+20,1.781755e+21,1360.688,82.495461,11064,-14.065587,-3.494758,10.264826,17.500038,0,5.602466e+20,29.608,127.948,56.867,1.981149,1.018732e+12,-0.025063,-0.005277,5.203,1.043649e+12,20.278,1.318,2010-05-02 12:00:00,2010,11064,NaN,0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,9999.0,0.000000e+00


In [ ]:
# ============================================================
# 6. Quality checks
# ============================================================

print("Final dataset shape:", sharp_48h_df.shape)
print("Date range:", sharp_48h_df["T_REC_dt"].min(), "to", sharp_48h_df["T_REC_dt"].max())
print("Unique NOAA ARs:", sharp_48h_df["NOAA_AR"].nunique())

print("\nLabel counts:")
display(sharp_48h_df["label_MX_48h"].value_counts().rename("count").to_frame())

print("\nLabel percentages:")
display((sharp_48h_df["label_MX_48h"].value_counts(normalize=True) * 100).rename("percent").to_frame())

history_cols = [
    "MX_count_last_24h",
    "MX_count_last_48h",
    "C_count_last_24h",
    "C_count_last_48h",
    "max_flare_flux_last_48h",
    "mean_flare_flux_last_48h",
    "time_since_last_MX_hours",
    "flare_activity_index_48h"
]

print("\nHistory feature summary:")
display(sharp_48h_df[history_cols].describe().T)

# Sanity check: positive examples.
positive_examples = sharp_48h_df[sharp_48h_df["label_MX_48h"] == 1].head(10)
print("\nPositive examples:")
display(positive_examples[["NOAA_AR", "T_REC_dt", "label_MX_48h"] + history_cols])


Final dataset shape: (22355, 40)
Date range: 2010-05-01 12:00:00 to 2025-07-16 12:00:00
Unique NOAA ARs: 2268

Label counts:


,count
label_MX_48h,
0,21187
1,1168



Label percentages:


,percent
label_MX_48h,
0,94.775218
1,5.224782



History feature summary:


,count,mean,std,min,25%,50%,75%,max
MX_count_last_24h,22355.0,0.062626,0.458257,0.000000,0.0,0.0,0.000000e+00,16.000000
MX_count_last_48h,22355.0,0.120599,0.787383,0.000000,0.0,0.0,0.000000e+00,24.000000
C_count_last_24h,22355.0,0.479937,1.587418,0.000000,0.0,0.0,0.000000e+00,24.000000
C_count_last_48h,22355.0,0.925117,2.820196,0.000000,0.0,0.0,0.000000e+00,46.000000
max_flare_flux_last_48h,22355.0,0.000003,0.000020,0.000000,0.0,0.0,8.000000e-07,0.000930
mean_flare_flux_last_48h,22355.0,0.000001,0.000004,0.000000,0.0,0.0,6.200000e-07,0.000180
time_since_last_MX_hours,22355.0,9495.884756,2183.585657,0.033333,9999.0,9999.0,9.999000e+03,9999.000000
flare_activity_index_48h,22355.0,0.000007,0.000044,0.000000,0.0,0.0,1.180000e-06,0.001677



Positive examples:


,NOAA_AR,T_REC_dt,label_MX_48h,MX_count_last_24h,MX_count_last_48h,C_count_last_24h,C_count_last_48h,max_flare_flux_last_48h,mean_flare_flux_last_48h,time_since_last_MX_hours,flare_activity_index_48h
99,11079,2010-06-12 12:00:00,1,0.0,0.0,0.0,0.0,3.800000e-07,3.800000e-07,9999.0,3.800000e-07
105,11081,2010-06-11 12:00:00,1,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,9999.0,0.000000e+00
208,11093,2010-08-06 12:00:00,1,0.0,0.0,0.0,0.0,2.800000e-07,2.800000e-07,9999.0,2.800000e-07
209,11093,2010-08-07 12:00:00,1,0.0,0.0,0.0,0.0,2.800000e-07,2.800000e-07,9999.0,2.800000e-07
401,11112,2010-10-15 12:00:00,1,0.0,0.0,0.0,0.0,3.900000e-07,3.050000e-07,9999.0,6.100000e-07
402,11112,2010-10-16 12:00:00,1,0.0,0.0,0.0,0.0,3.900000e-07,2.500000e-07,9999.0,7.500000e-07
736,11153,2011-02-07 12:00:00,1,0.0,0.0,0.0,0.0,1.000000e-07,1.000000e-07,9999.0,1.000000e-07
737,11153,2011-02-08 12:00:00,1,0.0,0.0,0.0,0.0,2.400000e-07,1.840000e-07,9999.0,9.200000e-07
766,11158,2011-02-12 12:00:00,1,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,9999.0,0.000000e+00
767,11158,2011-02-13 12:00:00,1,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,9999.0,0.000000e+00


In [ ]:
# ============================================================
# 7. Save output dataset
# ============================================================

sharp_48h_df.to_csv(OUTPUT_PATH, index=False)

print("Saved 48h ML-ready dataset to:")
print(OUTPUT_PATH)

# Optional: save a small metadata summary.
summary_path = OUTPUT_PATH.replace(".csv", "_summary.txt")

with open(summary_path, "w") as f:
    f.write("X-FlareXAI 48h Dataset Summary\n")
    f.write("=" * 40 + "\n")
    f.write(f"Rows: {len(sharp_48h_df)}\n")
    f.write(f"Columns: {sharp_48h_df.shape[1]}\n")
    f.write(f"Date range: {sharp_48h_df['T_REC_dt'].min()} to {sharp_48h_df['T_REC_dt'].max()}\n")
    f.write(f"Unique NOAA ARs: {sharp_48h_df['NOAA_AR'].nunique()}\n")
    f.write("\nLabel counts:\n")
    f.write(str(sharp_48h_df["label_MX_48h"].value_counts()) + "\n")
    f.write("\nLabel percentages:\n")
    f.write(str(sharp_48h_df["label_MX_48h"].value_counts(normalize=True) * 100) + "\n")

print("Saved summary to:")
print(summary_path)


Saved 48h ML-ready dataset to:
/content/drive/MyDrive/AR_Stratified/CICCE_BRADFORD_PROJECT/HMI_AR_2010_2025_ML_READY_48H.csv
Saved summary to:
/content/drive/MyDrive/AR_Stratified/CICCE_BRADFORD_PROJECT/HMI_AR_2010_2025_ML_READY_48H_summary.txt


# Next notebook

After this dataset is created, continue with:

**XFlareXAI_02_Models_and_XAI.ipynb**

That notebook should:

1. Load `HMI_AR_2010_2025_ML_READY_48H.csv`
2. Select SHARP + GOES history features
3. Use chronological train/validation/test split
4. Train XGBoost
5. Train MLP
6. Generate SHAP plots
7. Generate LIME explanations
8. Generate Integrated Gradients for MLP
9. Save results and figures for the paper
